# SIB - Extra: LogisticRegression com momentum e regularização L1

A regressão logística aplica a sigmoide à combinação linear das features:

$$h_\theta(x) = \frac{1}{1 + e^{-\theta^T x}}$$

e minimiza a cross-entropy regularizada:

$$J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log h_\theta(x^{(i)}) + (1 - y^{(i)}) \log (1 - h_\theta(x^{(i)})) \right] + \frac{\lambda_2}{2m} \sum_j \theta_j^2 + \frac{\lambda_1}{m} \sum_j |\theta_j|$$

**Momentum**: em vez de atualizar theta só com o gradiente atual, usa-se uma média móvel
exponencial dos gradientes, $v_t = \beta v_{t-1} + (1 - \beta) \nabla J$ e $\theta_t = \theta_{t-1} - \alpha v_t$
(convenção do `formulas.md` do curso). Isto acelera a descida em direções consistentes e
amortece as oscilações (ver secção 4).

**L1**: a sub-derivada de $|\theta_j|$ é $\text{sign}(\theta_j)$, mas usá-la diretamente faz os
coeficientes oscilar em torno de zero. Por isso a L1 é aplicada como um passo de
soft-thresholding, $\theta_j \leftarrow \text{sign}(\theta_j) \max(|\theta_j| - \alpha \lambda_1 / m, 0)$,
que os põe exatamente a zero (seleção de features), ao contrário da L2, que só os encolhe.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from si.io.csv_file import read_csv
from si.model_selection.split import stratified_train_test_split
from si.models.logistic_regression import LogisticRegression
from si.statistics.sigmoid_function import sigmoid_function

## 1. Função sigmoide

In [2]:
z = np.linspace(-10, 10, 200)

plt.figure(figsize=(6, 3.5))
plt.plot(z, sigmoid_function(z))
plt.axhline(0.5, color='grey', linestyle='--', linewidth=0.8)
plt.axvline(0, color='grey', linestyle='--', linewidth=0.8)
plt.xlabel('z')
plt.ylabel('sigmoid(z)')
plt.title('Função sigmoide')
plt.tight_layout()
plt.show()

print('sigmoid(0)    =', sigmoid_function(np.array([0.0]))[0])
print('sigmoid(-100) =', sigmoid_function(np.array([-100.0]))[0])
print('sigmoid(100)  =', sigmoid_function(np.array([100.0]))[0])

sigmoid(0)    = 0.5
sigmoid(-100) = 3.7200759760208356e-44
sigmoid(100)  = 1.0


## 2. Dataset breast-bin

Classificação binária: tumor benigno (0) vs maligno (1).

In [3]:
breast = read_csv('../datasets/breast_bin/breast-bin.csv', sep=',', features=False, label=True)
train, test = stratified_train_test_split(breast, test_size=0.2, random_state=42)

print('dimensões:', breast.shape())
print('classes:', breast.get_classes())
print('treino:', train.shape(), '| teste:', test.shape())

dimensões: (698, 9)
classes: [0. 1.]
treino: (559, 9) | teste: (139, 9)


## 3. Modelo base (só L2)

In [4]:
model = LogisticRegression(l2_penalty=1.0, alpha=0.1, max_iter=2000, patience=20, scale=True).fit(train)

print('iterações:', len(model.cost_history))
print('accuracy (treino): {:.4f}'.format(model.score(train)))
print('accuracy (teste):  {:.4f}'.format(model.score(test)))
print()
print('primeiras 5 probabilidades:', np.round(model.predict_proba(test)[:5], 4))
print('primeiras 5 classes:       ', model.predict(test)[:5])
print('primeiras 5 reais:         ', test.y[:5].astype(int))

iterações: 172
accuracy (treino): 0.9678
accuracy (teste):  0.9784

primeiras 5 probabilidades: [0.1103 0.0207 0.0727 0.0533 0.0255]
primeiras 5 classes:        [0 0 0 0 0]
primeiras 5 reais:          [0 0 0 0 0]


## 4. Efeito do momentum

A convenção usada é a do `formulas.md` do curso (média móvel exponencial):

$$v_t = \beta v_{t-1} + (1 - \beta) \nabla J \quad ; \quad \theta_t = \theta_{t-1} - \alpha v_t$$

Com $\beta = 0$ reduz-se ao gradient descent normal. Como o fator $(1 - \beta)$ normaliza a
atualização, o tamanho do passo não explode com betas altos — ao contrário da forma clássica
$v_t = \beta v_{t-1} + \nabla J$, em que o passo efetivo é amplificado por $1/(1-\beta)$.

In [5]:
print('Paragem por patience (alpha = 0.1, patience = 20)')
print('{:>10s} {:>12s} {:>14s} {:>14s}'.format('momentum', 'iterações', 'custo final', 'acc. teste'))

histories = {}
for beta in [0.0, 0.5, 0.9, 0.95, 0.99]:
    m = LogisticRegression(l2_penalty=1.0, alpha=0.1, max_iter=5000, patience=20,
                           scale=True, momentum=beta).fit(train)
    histories[beta] = list(m.cost_history.values())
    print('{:>10.2f} {:>12d} {:>14.6f} {:>14.4f}'.format(
        beta, len(m.cost_history), histories[beta][-1], m.score(test)))

Paragem por patience (alpha = 0.1, patience = 20)
  momentum    iterações    custo final     acc. teste
      0.00          172       0.102442         0.9784
      0.50          170       0.102505         0.9784
      0.90          148       0.103130         0.9784
      0.95          119       0.103412         0.9784
      0.99          129       0.107347         0.9784


Comparar pelo número de iterações até à paragem confunde duas coisas: a velocidade de
convergência e o critério de paragem. A comparação honesta é fixar o orçamento de iterações
e ver quem chega mais baixo.

In [6]:
print('Custo após um número fixo de iterações (patience desligada)')
print('{:>10s} {:>12s} {:>12s} {:>12s}'.format('momentum', 'N = 50', 'N = 200', 'N = 1000'))

for beta in [0.0, 0.5, 0.9, 0.95, 0.99]:
    row = []
    for n_iter in [50, 200, 1000]:
        m = LogisticRegression(l2_penalty=1.0, alpha=0.1, max_iter=n_iter, patience=n_iter + 1,
                               scale=True, momentum=beta).fit(train)
        row.append(list(m.cost_history.values())[-1])
    print('{:>10.2f} {:>12.5f} {:>12.5f} {:>12.5f}'.format(beta, *row))

Custo após um número fixo de iterações (patience desligada)
  momentum       N = 50      N = 200     N = 1000
      0.00      0.13485      0.10053      0.09084
      0.50      0.13370      0.10047      0.09084
      0.90      0.12320      0.09987      0.09083


      0.95      0.12068      0.09912      0.09081


      0.99      0.20809      0.11474      0.09080


In [7]:
plt.figure(figsize=(7, 4))
for beta, costs in histories.items():
    plt.plot(costs, label='momentum = {}'.format(beta))
plt.xlabel('iteração')
plt.ylabel('custo J')
plt.title('Convergência com e sem momentum')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Efeito da regularização L1

Com L2 = 0 e L1 a aumentar, os coeficientes são empurrados para zero — é isto que distingue
a L1 da L2, que apenas os encolhe sem os anular.

In [8]:
print('{:>10s} {:>12s} {:>14s} {:>12s}'.format('l1_penalty', 'sum|theta|', 'theta == 0', 'acc. teste'))
for l1 in [0.0, 10.0, 50.0, 100.0, 200.0, 500.0]:
    m = LogisticRegression(l2_penalty=0.0, l1_penalty=l1, alpha=0.1,
                           max_iter=3000, patience=50, scale=True).fit(train)
    n_zero = int(np.sum(m.theta == 0))
    print('{:>10.0f} {:>12.4f} {:>14d} {:>12.4f}'.format(
        l1, np.sum(np.abs(m.theta)), n_zero, m.score(test)))

l1_penalty   sum|theta|     theta == 0   acc. teste
         0       5.4168              0       0.9784
        10       4.0363              0       0.9784
        50       2.0787              1       0.9640
       100       1.1675              3       0.9353
       200       0.1615              7       0.6547
       500       0.0000              9       0.6547


In [9]:
l1_values = [0.0, 50.0, 100.0, 200.0]
fig, axes = plt.subplots(1, len(l1_values), figsize=(14, 3.2), sharey=True)
for ax, l1 in zip(axes, l1_values):
    m = LogisticRegression(l2_penalty=0.0, l1_penalty=l1, alpha=0.1,
                           max_iter=3000, patience=50, scale=True).fit(train)
    ax.bar(range(len(m.theta)), m.theta)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title('L1 = {:.0f}'.format(l1))
    ax.set_xlabel('feature')
axes[0].set_ylabel('theta')
plt.tight_layout()
plt.show()

## Conclusões

- Com um orçamento fixo de iterações, momentum de 0.9-0.95 chega a um custo mais baixo do que
  o gradient descent puro. O ganho aqui é modesto porque o problema é fácil e quase convexo
  sem vales estreitos; o momentum compensa mais em superfícies mal condicionadas.
- Momentum = 0.99 é contraproducente no início (custo mais alto a N = 50), porque a média móvel
  fica presa a gradientes antigos e ultrapassa o mínimo; só recupera com muitas iterações.
- A L1 anula coeficientes de forma exata graças ao soft-thresholding. Um sub-gradiente
  `sign(theta)` simples fá-los oscilar em torno de zero sem lá chegar.
- A L2, ao contrário da L1, nunca põe coeficientes exatamente a zero: apenas os encolhe.
- A accuracy só cai depois de a L1 remover features realmente informativas; com lambda
  demasiado alto o modelo colapsa para a classe maioritária (~0.65 neste dataset).